#### Загрузка данных и передобработка

In [1]:
import pandas as pd


In [2]:
df = pd.read_parquet("data/final_table.parquet")
print(df.shape)
df.head(5)
# Соотношение классов в целевой переменной
print(
    f"Target positive rate: {(df.target.value_counts()[1] / df.target.value_counts()[0]) * 100:.2f}%"
)


(4466001, 25)
Target positive rate: 13.58%


In [3]:
# Разделение на обучающую, валидационную и тестовую выборки по месяцам (октябрь- train, ноябрь - val, декабрь - test)
train_df = df[df["month"] == 10]
val_df = df[df["month"] == 11]
test_df = df[df["month"] == 12]
print(train_df.shape)
print(val_df.shape)
print(test_df.shape)
print((train_df.shape[0] + val_df.shape[0] + test_df.shape[0]) == df.shape[0])

(1517741, 25)
(1485926, 25)
(1462334, 25)
True


In [4]:
# Удаление ненужных признаков
cols_to_drop = ["timestamp", "user_id", "post_id", "target", "month"]

# Формирование признаков и целевой переменной X_train, X_test и y_train, y_test
X_train = train_df.drop(
    columns=cols_to_drop,
    axis=1,
)
X_val = val_df.drop(
    columns=cols_to_drop,
    axis=1,
)

X_test = test_df.drop(
    columns=cols_to_drop,
    axis=1,
)

y_train = train_df["target"]
y_val = val_df["target"]
y_test = test_df["target"]

In [5]:
# Проверка размеров и типов данных
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)
print(X_train.columns.equals(X_test.columns))
print(X_train.dtypes)
print(X_train.shape[0] + X_val.shape[0] + X_test.shape[0] == df.shape[0])

(1517741, 20) (1517741,)
(1485926, 20) (1485926,)
(1462334, 20) (1462334,)
True
hour              int32
dow               int32
gender            int64
age               int64
country          object
city             object
exp_group         int64
os               object
source           object
topic            object
tfidf_film      float32
tfidf_good      float32
tfidf_like      float32
tfidf_movie     float32
tfidf_mr        float32
tfidf_not       float32
tfidf_people    float32
tfidf_say       float32
tfidf_time      float32
tfidf_year      float32
dtype: object
True


In [6]:
# Очистка памяти и удаление временных переменных
del train_df, val_df, test_df

import gc

gc.collect()

0

In [7]:
print(X_train.select_dtypes(include=["object", "category"]).columns)
print(X_train.select_dtypes(exclude=["object", "category"]).columns)
print(X_train.select_dtypes(exclude=["object", "category"]).nunique())

Index(['country', 'city', 'os', 'source', 'topic'], dtype='object')
Index(['hour', 'dow', 'gender', 'age', 'exp_group', 'tfidf_film', 'tfidf_good',
       'tfidf_like', 'tfidf_movie', 'tfidf_mr', 'tfidf_not', 'tfidf_people',
       'tfidf_say', 'tfidf_time', 'tfidf_year'],
      dtype='object')
hour              18
dow                7
gender             2
age               63
exp_group          5
tfidf_film      1598
tfidf_good      1685
tfidf_like      1719
tfidf_movie     1606
tfidf_mr         743
tfidf_not       2018
tfidf_people    1344
tfidf_say       1711
tfidf_time      1679
tfidf_year      1400
dtype: int64


In [8]:
# Выделение категориальных признаков
category_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()
all_category_columns = category_features + ["gender", "exp_group"]
print(all_category_columns)

['country', 'city', 'os', 'source', 'topic', 'gender', 'exp_group']


In [9]:
# Преобразование категориальных признаков в строковый тип данных
for col in all_category_columns:
    X_train[col] = X_train[col].astype("str")
    X_val[col] = X_val[col].astype("str")
    X_test[col] = X_test[col].astype("str")

In [10]:
X_train.head(5)

,hour,dow,gender,age,country,city,exp_group,os,source,topic,tfidf_film,tfidf_good,tfidf_like,tfidf_movie,tfidf_mr,tfidf_not,tfidf_people,tfidf_say,tfidf_time,tfidf_year
282,14,4,0,45,Russia,Irkutsk,0,Android,ads,covid,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.00000,0.0
283,14,4,0,45,Russia,Irkutsk,0,Android,ads,covid,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.00000,0.0
284,14,4,0,45,Russia,Irkutsk,0,Android,ads,movie,0.296770,0.146551,0.148124,0.735463,0.201824,0.132843,0.0,0.274484,0.44032,0.0
285,14,4,0,45,Russia,Irkutsk,0,Android,ads,movie,0.872395,0.287204,0.000000,0.000000,0.395527,0.000000,0.0,0.000000,0.00000,0.0
286,14,4,0,45,Russia,Irkutsk,0,Android,ads,movie,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.00000,0.0


In [11]:
set(all_category_columns) - set(X_train.columns)

set()

In [12]:
X_train[all_category_columns].dtypes

country      object
city         object
os           object
source       object
topic        object
gender       object
exp_group    object
dtype: object

In [13]:
X_train.drop(columns=all_category_columns).dtypes

hour              int32
dow               int32
age               int64
tfidf_film      float32
tfidf_good      float32
tfidf_like      float32
tfidf_movie     float32
tfidf_mr        float32
tfidf_not       float32
tfidf_people    float32
tfidf_say       float32
tfidf_time      float32
tfidf_year      float32
dtype: object

#### Создание и обучение модели

In [15]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.1,
    depth=6,
    cat_features=all_category_columns,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="CPU",
)


In [ ]:
# # Ограничение размера выборки для тестирования обучения модели
# X_train_smoke = X_train.sample(n=200_000, random_state=42)
# y_train_smoke = y_train.loc[X_train_smoke.index]

# X_test_smoke = X_test.sample(n=200_000, random_state=42)
# y_test_smoke = y_test.loc[X_test_smoke.index]

# print(X_train_smoke.shape, y_train_smoke.shape)
# print(X_test_smoke.shape, y_test_smoke.shape)

(200000, 20) (200000,)
(200000, 20) (200000,)


In [16]:
model.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=20,
    use_best_model=True,
    verbose=20,
)

# Получение вероятностей положительного класса для обучающей, валидационной и тестовой выборок
train_proba = model.predict_proba(X_train)[:, 1]
val_proba = model.predict_proba(X_val)[:, 1]
test_proba = model.predict_proba(X_test)[:, 1]

0:	test: 0.5082077	best: 0.5082077 (0)	total: 1.53s	remaining: 5m 4s
20:	test: 0.6327090	best: 0.6327090 (20)	total: 18s	remaining: 2m 33s
40:	test: 0.6449529	best: 0.6449529 (40)	total: 33.9s	remaining: 2m 11s
60:	test: 0.6540250	best: 0.6540377 (59)	total: 52.5s	remaining: 1m 59s
80:	test: 0.6567578	best: 0.6567578 (80)	total: 1m 11s	remaining: 1m 45s
100:	test: 0.6580186	best: 0.6580186 (100)	total: 1m 31s	remaining: 1m 29s
120:	test: 0.6589016	best: 0.6589016 (120)	total: 1m 52s	remaining: 1m 13s
140:	test: 0.6595525	best: 0.6595525 (140)	total: 2m 11s	remaining: 55.1s
160:	test: 0.6604436	best: 0.6604436 (160)	total: 2m 31s	remaining: 36.8s
180:	test: 0.6609950	best: 0.6609950 (180)	total: 2m 52s	remaining: 18.1s
199:	test: 0.6616990	best: 0.6616990 (199)	total: 3m 10s	remaining: 0us

bestTest = 0.661699037
bestIteration = 199



In [17]:
from sklearn.metrics import roc_auc_score

train_roc_auc = roc_auc_score(y_train, train_proba)
val_roc_auc = roc_auc_score(y_val, val_proba)
test_roc_auc = roc_auc_score(y_test, test_proba)

print(f"Train ROC AUC: {train_roc_auc: .4f}")
print(f"Validation ROC AUC: {val_roc_auc: .4f}")
print(f"Test ROC AUC: {test_roc_auc: .4f}")
print("Лучшая итерация:", model.get_best_iteration())
print("Лучшая validation AUC:", model.get_best_score()["validation"]["AUC"])

Train ROC AUC:  0.7006
Validation ROC AUC:  0.6617
Test ROC AUC:  0.6551
Лучшая итерация: 199
Лучшая validation AUC: 0.6616990370364946


In [18]:
%%time

from catboost import CatBoostClassifier

model_exp = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    cat_features=all_category_columns,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="CPU",
)

model_exp.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    early_stopping_rounds=30,
    use_best_model=True,
    verbose=20,
)


0:	test: 0.5082077	best: 0.5082077 (0)	total: 1.11s	remaining: 9m 15s
20:	test: 0.6327090	best: 0.6327090 (20)	total: 15.2s	remaining: 5m 45s
40:	test: 0.6449529	best: 0.6449529 (40)	total: 31s	remaining: 5m 47s
60:	test: 0.6540250	best: 0.6540377 (59)	total: 49.1s	remaining: 5m 53s
80:	test: 0.6567578	best: 0.6567578 (80)	total: 1m 8s	remaining: 5m 52s
100:	test: 0.6580186	best: 0.6580186 (100)	total: 1m 27s	remaining: 5m 44s
120:	test: 0.6589016	best: 0.6589016 (120)	total: 1m 47s	remaining: 5m 35s
140:	test: 0.6595525	best: 0.6595525 (140)	total: 2m 5s	remaining: 5m 18s
160:	test: 0.6604436	best: 0.6604436 (160)	total: 2m 24s	remaining: 5m 4s
180:	test: 0.6609950	best: 0.6609950 (180)	total: 2m 43s	remaining: 4m 48s
200:	test: 0.6617354	best: 0.6617354 (200)	total: 3m 3s	remaining: 4m 32s
220:	test: 0.6622727	best: 0.6622727 (220)	total: 3m 24s	remaining: 4m 18s
240:	test: 0.6626244	best: 0.6626244 (240)	total: 3m 44s	remaining: 4m 1s
260:	test: 0.6629631	best: 0.6629631 (260)	total

In [19]:
# Получение вероятностей положительного класса для обучающей, валидационной и тестовой выборок
train_proba = model_exp.predict_proba(X_train)[:, 1]
val_proba = model_exp.predict_proba(X_val)[:, 1]
# test_proba = model_exp.predict_proba(X_test)[:, 1]

train_roc_auc_exp = roc_auc_score(y_train, train_proba)
val_roc_auc_exp = roc_auc_score(y_val, val_proba)
# test_roc_auc_exp = roc_auc_score(y_test, test_proba)

print(f"Train ROC AUC: {train_roc_auc_exp: .4f}")
print(f"Validation ROC AUC: {val_roc_auc_exp: .4f}")
# print(f"Test ROC AUC: {test_roc_auc_exp: .4f}")
print("Лучшая итерация:", model_exp.get_best_iteration())
print("Лучшая validation AUC:", model_exp.get_best_score()["validation"]["AUC"])

Train ROC AUC:  0.7095
Validation ROC AUC:  0.6662
Лучшая итерация: 499
Лучшая validation AUC: 0.6661766739922205


In [20]:
del train_proba, val_proba, test_proba, train_roc_auc, val_roc_auc, test_roc_auc
del model, model_exp

import gc

gc.collect()

0

In [21]:
import pandas as pd

X_final_train = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True,
)

y_final_train = pd.concat(
    [y_train, y_val],
    axis=0,
    ignore_index=True,
)

print(X_final_train.shape)
print(y_final_train.shape)
print(y_final_train.mean())

(3003667, 20)
(3003667,)
0.11087580613962866


In [22]:
final_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    cat_features=all_category_columns,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    task_type="CPU",
    verbose=50,
)

final_model.fit(X_final_train, y_final_train)


0:	total: 1.14s	remaining: 9m 29s
50:	total: 59.7s	remaining: 8m 45s
100:	total: 2m 4s	remaining: 8m 12s
150:	total: 3m 9s	remaining: 7m 17s
200:	total: 4m 15s	remaining: 6m 19s
250:	total: 5m 20s	remaining: 5m 18s
300:	total: 6m 28s	remaining: 4m 17s
350:	total: 7m 35s	remaining: 3m 13s
400:	total: 8m 41s	remaining: 2m 8s
450:	total: 9m 47s	remaining: 1m 3s
499:	total: 10m 51s	remaining: 0us


In [23]:
test_proba = final_model.predict_proba(X_test)[:, 1]
test_roc_auc = roc_auc_score(y_test, test_proba)

print(f"Final Test ROC AUC: {test_roc_auc:.4f}")

Final Test ROC AUC: 0.6620


In [24]:
# Сохранение модели в файл

import pickle
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "catboost_recommender.pkl"

with open(model_path, "wb") as file:
    pickle.dump(final_model, file)

print(f"Модель сохранена: {model_path.resolve()}")

Модель сохранена: F:\Prog\Py\KK_final_proj\models\catboost_recommender.pkl


In [25]:
with open(model_path, "rb") as file:
    loaded_model = pickle.load(file)

print(type(loaded_model))
print("predict:", hasattr(loaded_model, "predict"))
print("predict_proba:", hasattr(loaded_model, "predict_proba"))

<class 'catboost.core.CatBoostClassifier'>
predict: True
predict_proba: True


In [33]:
X_test = X_test[loaded_model.feature_names_]
check_data = X_test.head(10)
true_data = y_test.head(10).tolist()

check_predictions = loaded_model.predict(check_data)
check_probabilities = loaded_model.predict_proba(check_data)[:, 1]

print("Предсказания:")
print(check_predictions)

print("Истинные значения:")
print(true_data)

print("Вероятности лайка:")
print(check_probabilities)

Предсказания:
[0 0 0 0 0 0 0 0 0 0]
Истинные значения:
[0, 0, 0, 0, 1, 0, 0, 1, 0, 0]
Вероятности лайка:
[0.13503929 0.04926018 0.05178514 0.05894282 0.04829747 0.04225687
 0.13503929 0.13503929 0.04829827 0.05806543]


In [27]:
import numpy as np

original_proba = final_model.predict_proba(check_data)[:, 1]
loaded_proba = loaded_model.predict_proba(check_data)[:, 1]

print(
    "Предсказания совпадают:",
    np.allclose(original_proba, loaded_proba),
)

Предсказания совпадают: True


In [28]:
file_size_mb = model_path.stat().st_size / 1024**2

print(f"Размер модели: {file_size_mb:.2f} МБ")

Размер модели: 5.75 МБ


In [32]:
print(final_model.feature_names_)

['hour', 'dow', 'gender', 'age', 'country', 'city', 'exp_group', 'os', 'source', 'topic', 'tfidf_film', 'tfidf_good', 'tfidf_like', 'tfidf_movie', 'tfidf_mr', 'tfidf_not', 'tfidf_people', 'tfidf_say', 'tfidf_time', 'tfidf_year']


In [34]:
print(type(loaded_model))
print("predict:", callable(getattr(loaded_model, "predict", None)))
print(
    "predict_proba:",
    callable(getattr(loaded_model, "predict_proba", None)),
)

<class 'catboost.core.CatBoostClassifier'>
predict: True
predict_proba: True


#### Выделение признаков и сохранение признаков в таблицы

In [40]:
final_table = pd.read_parquet("data/final_table.parquet")

In [41]:
final_table.shape

(4466001, 25)

In [42]:
df_user_features = final_table[
    [
        "user_id",
        "age",
        "gender",
        "country",
        "city",
        "exp_group",
        "os",
        "source",
    ]
].drop_duplicates("user_id")

tfidf_columns = [
    column for column in final_table.columns if column.startswith("tfidf_")
]

df_post_features = final_table[
    [
        "post_id",
        "topic",
        *tfidf_columns,
    ]
].drop_duplicates("post_id")

In [43]:
print(df_user_features.shape)
print(df_user_features["user_id"].nunique())

print(df_post_features.shape)
print(df_post_features["post_id"].nunique())

(10588, 8)
10588
(6831, 12)
6831


In [ ]:
# Строка подключения к PostgreSQL
import os
from dotenv import load_dotenv

load_dotenv()

conn_str = os.getenv("DATABASE_URL")


In [46]:
# Сохраняем DataFrame в таблицу базы данных
df_user_features.to_sql(
    "vvtom_user_features",  # имя таблицы в БД
    conn_str,  # строка подключения
    if_exists="replace",  # перезапишет таблицу, если она есть
    index=False,  # не сохраняем индекс pandas в БД
    method="multi",  # ускоряем загрузку за счёт батчевой вставки
    chunksize=5000,  # размер чанка для пакетной вставки
)

df_post_features.to_sql(
    "vvtom_post_features",  # имя таблицы в БД
    conn_str,  # строка подключения
    if_exists="replace",  # перезапишет таблицу, если она есть
    index=False,  # не сохраняем индекс pandas в БД
    method="multi",  # ускоряем загрузку за счёт батчевой вставки
    chunksize=5000,  # размер чанка для пакетной вставки
)

6831

#### Проверка записи в базу данных

In [47]:
user_count_sql = pd.read_sql(
    """
    SELECT COUNT(*) AS rows_count
    FROM vvtom_user_features
    """,
    conn_str,
)

post_count_sql = pd.read_sql(
    """
    SELECT COUNT(*) AS rows_count
    FROM vvtom_post_features
    """,
    conn_str,
)

print("Локально пользователей:", len(df_user_features))
print("В SQL пользователей:", user_count_sql.loc[0, "rows_count"])

print("Локально постов:", len(df_post_features))
print("В SQL постов:", post_count_sql.loc[0, "rows_count"])

Локально пользователей: 10588
В SQL пользователей: 10588
Локально постов: 6831
В SQL постов: 6831


In [48]:
user_features_check = pd.read_sql(
    """
    SELECT *
    FROM vvtom_user_features
    LIMIT 5
    """,
    conn_str,
)

post_features_check = pd.read_sql(
    """
    SELECT *
    FROM vvtom_post_features
    LIMIT 5
    """,
    conn_str,
)

display(user_features_check)
display(post_features_check)

,user_id,age,gender,country,city,exp_group,os,source
0,15522,34,1,Russia,Krasnoyarsk,4,Android,ads
1,80687,23,1,Russia,Novosibirsk,3,Android,ads
2,80688,45,0,Russia,Irkutsk,0,Android,ads
3,15523,46,1,Russia,Sim,1,iOS,ads
4,80689,15,1,Russia,Lipetsk,4,iOS,ads


,post_id,topic,tfidf_film,tfidf_good,tfidf_like,tfidf_movie,tfidf_mr,tfidf_not,tfidf_people,tfidf_say,tfidf_time,tfidf_year
0,4012,covid,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,1483,sport,0.0,0.000000,0.318844,0.0,0.000000,0.0,0.000000,0.000000,0.947807,0.000000
2,1829,sport,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.000000,0.889328,0.317030,0.329527
3,1375,politics,0.0,0.397306,0.000000,0.0,0.273577,0.0,0.220127,0.186035,0.000000,0.827187
4,1638,sport,0.0,0.419034,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.629506,0.654319


In [49]:
user_duplicates_sql = pd.read_sql(
    """
    SELECT user_id, COUNT(*) AS count
    FROM vvtom_user_features
    GROUP BY user_id
    HAVING COUNT(*) > 1
    """,
    conn_str,
)

post_duplicates_sql = pd.read_sql(
    """
    SELECT post_id, COUNT(*) AS count
    FROM vvtom_post_features
    GROUP BY post_id
    HAVING COUNT(*) > 1
    """,
    conn_str,
)

print("Дубликаты пользователей:", len(user_duplicates_sql))
print("Дубликаты постов:", len(post_duplicates_sql))

Дубликаты пользователей: 0
Дубликаты постов: 0


модель сохранена в .pkl;
predict() и predict_proba() работают;
test ROC-AUC — 0.6620;
пользовательские и постовые признаки записаны в PostgreSQL;
количество строк после загрузки совпадает.